[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ZeruiW/frontier-ai-courses/blob/main/C65_ProblemSolving_Communication_Course/01_estimation/01_estimation.ipynb)

# 01 · 估算题与数量级心算（Fermi 估算 / 误差传播 / 标注-训练-显存-QPS 四案例 / 双路径校验）

目标：把"拍脑袋估一个数"变成**几个可以运行、可以断言的小工具**。

本 notebook 你会亲手实现：
1. **环境自检**
2. **Fermi 估算器** —— 把因子列表相乘得到点估计
3. **对数域误差传播器** —— 用 RSS 合成多个独立因子的误差，对比"最坏情况"与"独立情况"
4. **锚点数字表** —— 内置一份可查、可断言的锚点常数
5. **四个完整估算案例** —— 标注成本与工期 / 训练时长 / 显存 / QPS 与实例数，逐步实现
6. **达到目标召回需要多少数据** —— 幂律外推
7. **双路径交叉校验器** —— 判定两条独立估算是否互相印证

> 心智模型：**分解 → 锚点 → 相乘 → 校验，四步都做到，比算得精确更重要。**

## 0 · 环境自检

本课全程只用标准库 + numpy。没有 GPU 依赖、不联网、不下载数据。

In [ ]:
import sys, math
import numpy as np

print('Python :', sys.version.split()[0])
print('numpy  :', np.__version__)

assert sys.version_info >= (3, 8), '需要 Python 3.8+'
assert hasattr(np, 'sqrt')

print('\n环境自检通过：本课不需要 GPU、不需要联网。')

## 1 · Fermi 估算器：分解 → 锚点 → 相乘

把因子列表相乘，得到一个点估计。因子本身的选取（分解 + 锚点）是人做的事，
这个函数只负责"相乘"这一步——但正是这一步最容易因为单位不对齐而出错，所以我们还顺手做单位检查。

In [ ]:
def fermi_estimate(factors):
    """factors: 一串已经对齐单位的因子。返回它们的乘积（点估计）。"""
    p = 1.0
    for f in factors:
        p *= f
    return p

# 例子：一支 500 辆车的测试车队，每车每天跑 320 公里，平均每公里遇到 5 个交通标志，
# 一天总共会记录到多少个标志实例？
fleet_size, km_per_day, signs_per_km = 500, 320, 5
total_signs = fermi_estimate([fleet_size, km_per_day, signs_per_km])
assert total_signs == 800_000.0, total_signs

print(f'车队 {fleet_size} 辆 × {km_per_day} 公里/天 × {signs_per_km} 标志/公里 = {total_signs:,.0f} 个标志/天')
print('\n三个因子里换任何一个数量级都会让结果差 10 倍——这就是为什么锚点数字要背对量级，而不是背对具体数值。')

## 2 · 对数域误差传播：为什么不是简单相加

每个因子的误差用"以 2 为底的对数误差" $\sigma_i$ 表示（$\sigma=1$ 表示落在 0.5x-2x 之间）。
独立因子的误差按方差可加性（RSS）合成，而不是线性相加。

In [ ]:
def log_error_rss(sigmas_bits):
    """sigmas_bits: 每个因子的对数误差（比特，以2为底）。假设相互独立，返回合成后的总误差（比特）。"""
    return math.sqrt(sum(s * s for s in sigmas_bits))

def uncertainty_factor(sigma_bits):
    """把比特数的误差换算成"乘除倍数"。"""
    return 2 ** sigma_bits

sigmas = [1, 1, 1, 1, 1]                      # 五个因子，各自误差在 2x 以内
worst_case = 2 ** sum(sigmas)                 # 最坏情况：误差同向叠加，线性相加
independent = uncertainty_factor(log_error_rss(sigmas))   # 独立假设：RSS 合成

assert worst_case == 32
assert abs(log_error_rss(sigmas) - math.sqrt(5)) < 1e-9
assert 4.5 < independent < 5.0, independent

print(f'五个因子各自误差 2x 以内：')
print(f'  最坏情况（同向叠加）  -> 总误差 ×{worst_case}')
print(f'  独立假设（RSS 合成）  -> 总误差 ×{independent:.1f}')
print('\n面试里该报的是后一个数字（连同"假设独立"这句话），而不是前一个吓人的 32 倍。')

## 3 · 锚点数字表

数量级正确就够，不要求精确到个位。内置一份可查的常数表，并用"排序关系"而不是"精确数值"做断言——
因为这些数字本身就是近似值，断言排序关系比断言小数点后几位更符合它的本质。

In [ ]:
ANCHORS_GPU_TFLOPS = {'V100': 125, 'A100_80G': 300, 'H100_SXM': 1000}
ANCHORS_GPU_BANDWIDTH_GBs = {'V100': 900, 'A100_80G': 2000, 'H100_SXM': 3300}
ANCHORS_LABELING = {
    'price_per_box_cny': 0.8,
    'boxes_per_person_day': 1000,
    'seg_images_per_person_day': 40,
}
ANCHORS_DATA = {
    'image_1080p_jpeg_kb': 300,
    'image_1080p_raw_mb': 6.2,
    'gigabit_eth_mbps': 1000,
    'lte_uplink_mbps': 30,
}

# 断言的是数量级关系，而不是精确到个位的数值——这才是锚点数字该有的用法
assert ANCHORS_GPU_TFLOPS['H100_SXM'] > ANCHORS_GPU_TFLOPS['A100_80G'] > ANCHORS_GPU_TFLOPS['V100']
assert ANCHORS_GPU_BANDWIDTH_GBs['H100_SXM'] > ANCHORS_GPU_BANDWIDTH_GBs['A100_80G']
assert ANCHORS_LABELING['boxes_per_person_day'] > ANCHORS_LABELING['seg_images_per_person_day']
assert ANCHORS_DATA['image_1080p_raw_mb'] * 1024 > ANCHORS_DATA['image_1080p_jpeg_kb']   # 未压缩确实比压缩大

for name, d in [('GPU 算力 (TFLOPS)', ANCHORS_GPU_TFLOPS), ('GPU 显存带宽 (GB/s)', ANCHORS_GPU_BANDWIDTH_GBs)]:
    print(name, '->', d)
print('\n✅ 锚点表就位：记的是量级和排序（H100 > A100 > V100），不是精确到个位的参数表。')

## 4 · 案例①：标注成本与工期

$T_{\text{人天}} = N_{\text{图像}} \times k_{\text{框/图}} / r_{\text{框/人天}}$，
$T_{\text{日历天}} = T_{\text{人天}} / n_{\text{团队人数}}$，
$C = N_{\text{图像}} \times k_{\text{框/图}} \times p_{\text{单价}}$。

In [ ]:
def labeling_estimate(n_images, boxes_per_image, team_size,
                      boxes_per_person_day=ANCHORS_LABELING['boxes_per_person_day'],
                      price_per_box=ANCHORS_LABELING['price_per_box_cny']):
    """返回 (总框数, 总人天, 日历工期天数, 总成本元)。"""
    total_boxes = n_images * boxes_per_image
    person_days = total_boxes / boxes_per_person_day
    calendar_days = person_days / team_size
    cost = total_boxes * price_per_box
    return total_boxes, person_days, calendar_days, cost

# TSR 例子：10 万张图，每图 3 个标志框，10 人团队
boxes, person_days, days, cost = labeling_estimate(100_000, 3, 10)
assert boxes == 300_000
assert person_days == 300.0
assert days == 30.0
assert cost == 240_000.0

print(f'总框数 {boxes:,.0f}，总人天 {person_days:,.0f}，日历工期 {days:.0f} 天，总成本 ¥{cost:,.0f}')
print('\n四个数字缺一不可：面试官期待的是完整的一套答案，不是只报其中一个。')

## 5 · 案例②：训练时长

$T_{\text{秒}} = N_{\text{图像}} \times E_{\text{epoch}} / \Phi_{\text{吞吐}}$。

In [ ]:
def training_hours(n_images, epochs, throughput_img_per_sec):
    total_images_seen = n_images * epochs
    seconds = total_images_seen / throughput_img_per_sec
    return seconds / 3600

hours = training_hours(100_000, 24, 300)
assert abs(hours - 2.2222222) < 1e-4, hours
print(f'10 万张图 × 24 epoch ÷ 300 图/秒 = {hours:.2f} 小时')

# 吞吐打七折(真实训练常见的 IO/增强瓶颈)会怎样？
hours_derated = training_hours(100_000, 24, 300 * 0.7)
assert hours_derated > hours
print(f'吞吐打七折后：{hours_derated:.2f} 小时（约变成 {hours_derated/hours:.2f} 倍）')
print('\n✅ 吞吐是最容易被高估的因子——报数字时说清是实测还是理论峰值。')

## 6 · 案例③：显存

$M_{\text{总}} \approx N_{\text{参数}} \times (b_w + b_{\text{优化器}}) + A_{\text{激活}}$。
AdamW 混合精度典型配置：fp16 权重 2 字节 + fp32 master 4 字节 + 一阶矩 4 字节 + 二阶矩 4 字节 = 14 字节/参数。

In [ ]:
def gpu_memory_gb(n_params, bytes_per_param=14, activation_gb=2.0):
    param_bytes = n_params * bytes_per_param
    return param_bytes / 1024**3 + activation_gb

mem_adamw = gpu_memory_gb(50_000_000)                          # AdamW: 14 字节/参数
mem_sgd = gpu_memory_gb(50_000_000, bytes_per_param=2)         # 纯 fp16 SGD 无动量: 2 字节/参数

assert abs(mem_adamw - 2.6519) < 1e-3, mem_adamw
assert abs(mem_sgd - 2.0931) < 1e-3, mem_sgd
assert mem_adamw > mem_sgd

print(f'5 千万参数模型，AdamW 混合精度显存 ≈ {mem_adamw:.2f} GB')
print(f'同样模型，纯 SGD（无动量）显存    ≈ {mem_sgd:.2f} GB')
print(f'仅优化器状态差异就多占了 {(mem_adamw - mem_sgd):.2f} GB。')

## 7 · 案例④：QPS 与实例数

$\text{QPS}_{\text{单实例}} = \text{并发数} / \text{单次延迟}$，
$N_{\text{实例}} = \lceil \text{QPS}_{\text{目标}} / \text{QPS}_{\text{单实例}} \rceil$。

In [ ]:
def instances_needed(target_qps, latency_sec, concurrency_per_instance=1):
    qps_per_instance = concurrency_per_instance / latency_sec
    return math.ceil(target_qps / qps_per_instance)

n = instances_needed(1000, 0.02, concurrency_per_instance=4)
assert n == 5, n
print(f'目标 1000 QPS，单次延迟 20ms，每实例并发 4 -> 需要 {n} 台实例')

# 延迟翻倍(比如换成更大的模型)会怎样？
n2 = instances_needed(1000, 0.04, concurrency_per_instance=4)
assert n2 == 10
print(f'延迟翻倍到 40ms -> 需要 {n2} 台实例（翻倍关系，符合公式里延迟和实例数成正比）')

## 8 · 达到目标召回需要多少数据：幂律外推

$(1-R_{\text{目标}})/(1-R_0) = (N_0/N_{\text{目标}})^{\alpha}$，
解出 $N_{\text{目标}} = N_0 \cdot ((1-R_0)/(1-R_{\text{目标}}))^{1/\alpha}$。

In [ ]:
def data_needed_for_recall(n0, r0, target_recall, alpha=0.4):
    ratio = (1 - r0) / (1 - target_recall)
    return n0 * ratio ** (1 / alpha)

n_target = data_needed_for_recall(5000, 0.70, 0.90, alpha=0.4)
assert 77_000 < n_target < 79_000, n_target
print(f'试点 5000 张 @ 70% 召回，目标 90% 召回，alpha=0.4 -> 需要约 {n_target:,.0f} 张')
print(f'数据量要变成原来的 {n_target/5000:.1f} 倍——这就是幂律边际收益递减的直观体现。')

## 9 · 双路径交叉校验器

两条完全独立、依赖不同假设的路径，如果答案量级接近，可信度大幅提高；
差一个数量级以上，说明至少有一个假设错了，要回头检查。

In [ ]:
def cross_check(estimate_a, estimate_b, tolerance=3.0):
    """返回 (是否可信, 两者比值)。比值 <= tolerance 判定为可信。"""
    lo, hi = min(estimate_a, estimate_b), max(estimate_a, estimate_b)
    ratio = hi / lo if lo > 0 else float('inf')
    return ratio <= tolerance, ratio

# 路径 A：上面幂律外推得到的 n_target ≈ 78,000 张
# 路径 B：按标注预算反推，预算 8 万元，单价 0.8 元/框，每图 3 个框
budget_cny, price_per_box, boxes_per_image = 80_000, 0.8, 3
path_b = budget_cny / price_per_box / boxes_per_image

ok, ratio = cross_check(n_target, path_b, tolerance=3.0)
assert ok is True, (n_target, path_b, ratio)
print(f'路径 A（幂律外推）≈ {n_target:,.0f} 张，路径 B（预算反推）≈ {path_b:,.0f} 张')
print(f'比值 {ratio:.2f} <= 3.0 -> 可信：两条独立路径互相印证。')

# 如果路径 B 是一个数量级偏差很大的错误估计呢？
ok_bad, ratio_bad = cross_check(n_target, 900_000, tolerance=3.0)
assert ok_bad is False
print(f'\n若路径 B 给出 900,000 张，比值 {ratio_bad:.1f} > 3.0 -> 不可信，需要回头检查假设。')

## ✏️ 练习 1：存储需求估算器

实现 `storage_needed_tb(n_images, avg_size_kb)`：返回存储 N 张图像需要多少 TB
（$1\,\text{TB} = 1024^3\,\text{KB}$）。

In [ ]:
def storage_needed_tb(n_images, avg_size_kb):
    # TODO
    raise NotImplementedError

In [ ]:
# —— 练习 1 自测 ——
s1 = storage_needed_tb(1_000_000, 300)
assert 0.27 < s1 < 0.29, s1

s2 = storage_needed_tb(10_000_000, 300)
assert abs(s2 - 10 * s1) < 1e-9, (s1, s2)      # 图片数变 10 倍，存储线性变 10 倍

print(f'100 万张 1080p 压缩图 ≈ {s1:.3f} TB')
print(f'1000 万张同样的图     ≈ {s2:.3f} TB（10 倍关系）')
print('\n✅ 练习 1 通过：存储需求 = 图像数 × 单张大小，和 QPS/显存用的是同一套"数量 × 单位消耗"骨架。')

## ✏️ 练习 2：车载多摄像头带宽估算器

实现 `bandwidth_mbps(n_cameras, fps, image_size_kb)`：返回 N 路摄像头、每路 fps 帧/秒、
每帧 image_size_kb 大小时，总共需要多少 Mbps 带宽（$1\,\text{KB}=1024\,\text{字节}=8192\,\text{比特}$）。

In [ ]:
def bandwidth_mbps(n_cameras, fps, image_size_kb):
    # TODO
    raise NotImplementedError

In [ ]:
# —— 练习 2 自测 ——
bw = bandwidth_mbps(6, 30, 200)
assert abs(bw - 294.912) < 1e-6, bw

bw2 = bandwidth_mbps(6, 60, 200)
assert abs(bw2 - 2 * bw) < 1e-6      # 帧率翻倍，带宽线性翻倍

print(f'6 路摄像头 × 30fps × 200KB/帧 = {bw:.1f} Mbps')
print(f'帧率翻倍到 60fps            = {bw2:.1f} Mbps')
print('\n✅ 练习 2 通过：这正是本模块「存储与带宽」一节说的——和 QPS/显存同一套心算骨架。')

## ✏️ 练习 3：把点估计换算成置信区间

实现 `estimate_range(midpoint, sigma_bits)`：给定一个 Fermi 点估计 `midpoint` 和它的对数误差
`sigma_bits`（以 2 为底），返回 `(low, high) = (midpoint / 2**sigma_bits, midpoint * 2**sigma_bits)`。

In [ ]:
def estimate_range(midpoint, sigma_bits):
    # TODO
    raise NotImplementedError

In [ ]:
# —— 练习 3 自测 ——
assert estimate_range(100, 1) == (50.0, 200.0)
assert estimate_range(1000, 2) == (250.0, 4000.0)

lo, hi = estimate_range(total_signs, log_error_rss([1, 1, 1]))
print(f'车队每日标志数点估计 {total_signs:,.0f}，三因子各 2x 误差、独立假设下的区间：'
      f'[{lo:,.0f}, {hi:,.0f}]')
print('\n✅ 练习 3 通过：把第 2 节的"误差比特数"落地成一个真正能报给面试官的区间。')

---
### 📖 参考答案（先自己做，再对照）

In [ ]:
# 练习 1 参考答案
def storage_needed_tb(n_images, avg_size_kb):
    return n_images * avg_size_kb / 1024 ** 3

In [ ]:
# 练习 2 参考答案
def bandwidth_mbps(n_cameras, fps, image_size_kb):
    bits_per_sec = n_cameras * fps * image_size_kb * 1024 * 8
    return bits_per_sec / 1e6

In [ ]:
# 练习 3 参考答案
def estimate_range(midpoint, sigma_bits):
    factor = 2 ** sigma_bits
    return midpoint / factor, midpoint * factor

---
## 🧪 真实工程胶囊：面试估算心算清单 + 锚点速查（中英对照）

In [ ]:
RECIPE = r'''
# ══════════════════════════════════════════════════════════════════════
# A. 面试中读到一个估算题后的心算顺序
# ══════════════════════════════════════════════════════════════════════
# □ 先分解：这个量能不能写成 2-5 个可独立估计因子的乘积？
# □ 再锚点：每个因子有没有一个你能背出来的量级数字？
# □ 再相乘：口算乘出点估计，先对齐单位（尤其是时间单位：秒/小时/天）
# □ 最后校验：换一条独立路径重新算一遍，两者相差在 3-5 倍内才算通过

# ══════════════════════════════════════════════════════════════════════
# B. 常背锚点（数量级，不是精确值）
# ══════════════════════════════════════════════════════════════════════
# GPU FP16 算力：V100 ~125TFLOPS，A100 ~300TFLOPS，H100 千 TFLOPS 量级
# 标注：检测框约 1000 框/人天，单价约 0.5-1 元/框
# 图像：1080p 压缩约 300KB，未压缩约 6MB
# 网络：千兆网 1Gbps=125MB/s，5G 约百 Mbps 到 1Gbps

# ══════════════════════════════════════════════════════════════════════
# C. 口播模板（中 / EN）
# ══════════════════════════════════════════════════════════════════════
# CN: 「我先把这个问题分解成三个因子，每个我给一个锚点估计，乘起来大概是这个量级，
#      我再换一条路径校验一下。」
# EN: "Let me break this into three factors, anchor each with a rough number,
#      multiply them to get a point estimate, then sanity-check with an independent path."

# ══════════════════════════════════════════════════════════════════════
# D. 与其他模块 / 课程的分工
# ══════════════════════════════════════════════════════════════════════
# · 系统设计里的完整容量估算（降级方案/监控）-> C63 模块 04，本课只给通用估算方法与锚点表
# · 长尾数据的定向采集策略 -> C58，本课「数据需求估算」只给数量级方法
'''
print(RECIPE)
for token in ['分解', '锚点', '相乘', '校验', 'C63 模块 04', 'C58']:
    assert token in RECIPE, token
print('检查单覆盖：估算心算顺序 / 常背锚点 / 中英口播 / 与其他课程的分工')

### 小结

- **Fermi 估算法四步**：分解（写成因子乘积）→ 锚点（给每个因子一个量级数字）→
  相乘（对齐单位）→ 校验（换一条独立路径重算一遍）。跳过校验，等于跳过测试。
- **误差在对数域按方差可加性合成**：五个因子各自误差 2x 以内，最坏情况是 32 倍，
  但独立假设下只有约 4.7 倍——**面试里该报的是后一个数字，并说明"假设独立"**。
- **锚点数字只需要记量级和排序**（H100 > A100 > V100 的算力，检测框标注约 1000 框/人天，
  1080p 压缩图约 300KB），不需要精确到个位。
- **四类 ML 场景估算共用同一套"数量 × 单位消耗 ÷ 单位时间/容量"骨架**：
  标注（框数 ÷ 人天产出）、训练（图像总数 ÷ 吞吐）、显存（参数 × 字节数 + 激活）、
  QPS（目标 QPS ÷ 单实例 QPS）——存储和带宽也是同一个骨架的延伸。
- **达到目标召回需要多少数据，可以用幂律外推给出量级**，但更重要的是用第二条独立路径
  （比如标注预算反推）交叉校验——两条路径吻合，估计才可信。
- **三类系统性错误会让估算离谱**：隐藏常数（真实成本常是理论值的 1.5-3 倍）、
  非线性（临界点附近线性外推会系统性低估）、长尾（整体均值会被头部类别主导，稀有类必须单独估）。
- **与 C63-04 的分工**：本课是通用 Fermi 估算方法论，C63-04 是系统设计里的完整容量估算
  （含降级方案与监控），两者互补，不重复。

下一站：**模块 02 · 诊断与归因推理** —— 把"假设与验证"这一步做深，
学会用信息增益最大的策略，把"可能是数据问题"这类模糊猜测，变成可以被系统性排除的假设树。